# 02 Descriptives

Descriptive tables for Memo 1. Writes CSVs to outputs/tables/.
Descriptive figures for Memo 1. Reads outputs/tables, writes PNGs to outputs/figures/.

In [ ]:
%run 00_utils.ipynb

import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style(font_size=12)

## Tables

In [ ]:
OUT = ROOT / "outputs" / "tables"
pqr = pd.read_csv(ROOT / "data" / "processed" / "pqr_medicines_clean.csv")
pqr["manufacturer"] = pqr["firm"]   # harmonized firm names (see firm_crosswalk.csv)
fpp = pd.read_csv(ROOT / "data" / "processed" / "pq_fpp_clean.csv")
api = pd.read_csv(ROOT / "data" / "processed" / "pq_api_clean.csv")

In [ ]:
# T1 dataset overview
rows = []
for name, d, unit, firm, prod, ycol in [
    ("Global Fund PQR (medicines)", pqr, "transaction", "manufacturer", "product", "year"),
    ("WHO PQ finished products", fpp, "prequalified product", "firm", "INN, Dosage Form and Strength", "year"),
    ("WHO PQ active ingredients", api, "prequalified API", "firm", "INN", "year")]:
    rows.append({"dataset": name, "unit": unit, "n_obs": len(d),
                 "n_firms": d[firm].nunique(), "n_products": d[prod].nunique(),
                 "first_year": int(d[ycol].min()), "last_year": int(d[ycol].max())})
t1 = pd.DataFrame(rows); t1.to_csv(f"{OUT}/t1_dataset_overview.csv", index=False)
print(t1.to_string(index=False))

PQR gave me 74,689 medicine transactions from 110 firms and 128 products. The WHO lists added 655 finished products and 194 active ingredients. PQR is pooled over 2005-2026, but 2025-2026 are incomplete because of reporting lags.

In [ ]:
# T2 PQR by therapeutic area
t2 = pqr.groupby("area").agg(transactions=("pk", "size"),
                             spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                             products=("product", "nunique"),
                             manufacturers=("manufacturer", "nunique"),
                             countries=("country", "nunique")).round(1)
t2.to_csv(f"{OUT}/t2_pqr_by_area.csv"); print("\n", t2.to_string())

HIV dominated spend, at USD 8.3 bn of 11.7 bn. TB had the most products (60) and manufacturers (62), but its spend was about the same as malaria's. Spend alone would therefore understate how much of the supplier network is in TB.

In [ ]:
# T3 supplier concentration per product (spend-based HHI, 0-10000)
g = pqr.groupby(["area", "product", "manufacturer"])["cost_usd"].sum().reset_index()
def conc(x):
    s = x["cost_usd"] / x["cost_usd"].sum()
    return pd.Series({"n_manufacturers": len(x), "spend_usd_m": x["cost_usd"].sum() / 1e6,
                      "top1_share": s.max(), "hhi": (s ** 2).sum() * 10000,
                      "top_manufacturer": x.loc[x["cost_usd"].idxmax(), "manufacturer"]})
t3 = g.groupby(["area", "product"]).apply(conc, include_groups=False).reset_index()
t3["vulnerable_le2"] = t3["n_manufacturers"] <= 2
t3 = t3.sort_values("spend_usd_m", ascending=False)
t3.round(3).to_csv(f"{OUT}/t3_product_concentration.csv", index=False)
print("\nproducts:", len(t3), "| <=2 manufacturers:", t3.vulnerable_le2.sum(),
      "| HHI>2500:", (t3.hhi > 2500).sum())
print("spend share in <=2-mfr products:",
      round(t3.loc[t3.vulnerable_le2, "spend_usd_m"].sum() / t3.spend_usd_m.sum(), 3))
print(t3.groupby("area")[["n_manufacturers", "hhi"]].median().round(0).to_string())
print("\nlargest-spend vulnerable products:")
print(t3[t3.vulnerable_le2].head(10)[["area","product","n_manufacturers","spend_usd_m","top_manufacturer"]].round(1).to_string(index=False))

57 of 128 products (45%) had only 1 or 2 manufacturers over the whole period. These were mostly niche or newer products, such as delamanid and pretomanid, and they made up only 2.8% of spend. So thin supply is common across products but small in dollars. Median HHI was high in all three areas (HIV 5,152; TB 7,327; malaria 7,949). These HHIs pool all years, so a product could look less concentrated than it was in any single year.

In [ ]:
# T4 manufacturer breadth and spend
m = pqr.groupby("manufacturer").agg(products=("product", "nunique"),
                                    areas=("area", "nunique"),
                                    transactions=("pk", "size"),
                                    spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                                    first_year=("year", "min"), last_year=("year", "max"))
m["spend_share"] = m.spend_usd_m / m.spend_usd_m.sum()
m = m.sort_values("spend_usd_m", ascending=False)
m.round(3).to_csv(f"{OUT}/t4_manufacturers.csv")
print("\nmanufacturers:", len(m), "| top5 spend share:", round(m.spend_share.head(5).sum(), 3),
      "| top10:", round(m.spend_share.head(10).sum(), 3),
      "| in >=2 areas:", (m.areas >= 2).sum(), "| all 3 areas:", (m.areas == 3).sum())
print(m.head(10)[["products","areas","spend_usd_m","spend_share"]].round(3).to_string())

Supply was concentrated among a few firms. The top 5 held 62% of spend and the top 10 held 79%, and Mylan alone held 25%. Only 5 firms supplied all three disease areas. Guilin was the only Chinese firm in the top 10 (4.3%, malaria only).

In [ ]:
# T5 annual series (for figures)
yr = pqr[pqr.year.between(2008, 2025)]
t5 = yr.groupby(["year", "area"]).agg(transactions=("pk", "size"),
                                      spend_usd_m=("cost_usd", lambda x: x.sum() / 1e6),
                                      active_manufacturers=("manufacturer", "nunique")).reset_index()
t5.round(1).to_csv(f"{OUT}/t5_pqr_annual_by_area.csv", index=False)
t6 = fpp.dropna(subset=["year"]).groupby(["year", "Therapeutic Area"]).size().rename("n").reset_index()
t6.to_csv(f"{OUT}/t6_pq_fpp_annual_by_area.csv", index=False)

## Figures

In [ ]:
T, F = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures"
INK, INK2, MUTED, GRID = "#0b0b0b", "#595959", "#8a8984", "#e6e5e1"
S = {"HIV/ARV": "#003D78", "Tuberculosis": "#117996", "Malaria": "#F4B71B", "Other": "#B4B5B8"}

def title(ax, t, sub):
    ax.set_title(t, loc="left", fontsize=14, fontweight="bold", color=INK, pad=24)
    ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=11, color=INK2, va="bottom")

def source(fig, s):
    fig.text(0.01, -0.06, s, fontsize=9.5, color=MUTED, ha="left")

In [ ]:
# F1 annual PQR spend by area
a = pd.read_csv(T / "t5_pqr_annual_by_area.csv")
fig, ax = plt.subplots(figsize=(6.5, 4))
for area in ["HIV/ARV", "Tuberculosis", "Malaria"]:
    d = a[a.area == area]
    ax.plot(d.year, d.spend_usd_m, color=S[area], lw=2, marker="o", ms=3.5, solid_capstyle="round")
    # 2025 end points of TB and malaria sit close together, so spread their labels
    nudge = {"HIV/ARV": 30, "Tuberculosis": -35, "Malaria": 35}[area]
    ax.text(d.year.iloc[-1] + 0.3, d.spend_usd_m.iloc[-1] + nudge, area, color=INK, fontsize=11, va="center")
ax.set_xlim(2007.5, 2027.3); ax.set_ylim(0); ax.grid(axis="x", visible=False)
ax.set_xticks(range(2008, 2026, 2)); ax.set_ylabel("Spend per year (USD million)")
title(ax, "Global Fund medicine spending by disease area, 2008-2025",
      "Total product cost reported to PQR, by purchase order year")
source(fig, "Source: Global Fund Price & Quality Reporting, downloaded 2026-10-10. 2025 may be incomplete due to reporting lags.")
fig.savefig(F / "f1_pqr_spend_by_year.png"); plt.show()

HIV transactions peaked in 2013 and then fell as regimens moved to fewer fixed-dose combinations (t5). TB passed HIV in transaction counts in 2023. The last years in the figure should be read with care because of reporting lags.

In [ ]:
# F2 manufacturers per product
c = pd.read_csv(T / "t3_product_concentration.csv")
cnt = c.n_manufacturers.clip(upper=12).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6.5, 4))
cols = ["#7C1220" if k <= 2 else S["HIV/ARV"] for k in cnt.index]
ax.bar(cnt.index, cnt.values, color=cols, width=0.8, edgecolor="white", linewidth=1)
for k, v in cnt.items():
    ax.text(k, v + 0.4, str(v), ha="center", fontsize=10.5, color=INK2)
ax.set_xticks(cnt.index, [str(k) if k < 12 else "12+" for k in cnt.index])
ax.set_xlabel("Number of manufacturers that supplied the product"); ax.set_ylabel("Number of products")
ax.grid(axis="x", visible=False)
n2 = int((c.n_manufacturers <= 2).sum())
ax.annotate(f"{n2} of {len(c)} products have\n1 or 2 manufacturers", xy=(2.3, cnt.loc[2] + 1),
            xytext=(4.2, cnt.max() * 0.85), fontsize=11, color=INK,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))
title(ax, "Many Global Fund medicines rely on one or two suppliers",
      "Distinct manufacturers per product, all PQR medicine transactions 2005-2026")
source(fig, "Source: Global Fund PQR. Products are PQR product names (formulation level); manufacturers harmonized to firms.")
fig.savefig(F / "f2_manufacturers_per_product.png"); plt.show()

In [ ]:
# F3 manufacturer spend concentration
m = pd.read_csv(T / "t4_manufacturers.csv").sort_values("spend_usd_m", ascending=False)
top = m.head(10).copy()
short = {"Guilin Pharmaceutical": "Guilin (Fosun)"}
top["label"] = top.manufacturer.map(short).fillna(top.manufacturer)
rest = pd.DataFrame({"label": [f"Other {len(m) - 10} manufacturers"], "spend_share": [m.spend_share.iloc[10:].sum()]})
d = pd.concat([top[["label", "spend_share"]], rest]).iloc[::-1]
fig, ax = plt.subplots(figsize=(6.5, 4.6))
colors = [S["Other"] if l.startswith("Other") else S["HIV/ARV"] for l in d.label]
ax.barh(d.label, d.spend_share * 100, color=colors, height=0.72, edgecolor="white", linewidth=1)
for i, v in enumerate(d.spend_share * 100):
    ax.text(v + 0.4, i, f"{v:.1f}%", va="center", fontsize=10.5, color=INK2)
ax.set_xlabel("Share of total medicine spend (%)"); ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", length=0)
title(ax, "Ten firms account for three quarters of medicine spend",
      f"Share of USD {m.spend_usd_m.sum()/1000:.1f} bn in PQR medicine purchases, 2005-2026")
source(fig, "Source: Global Fund PQR. Total Product Cost (USD); excludes freight.")
fig.savefig(F / "f3_manufacturer_spend_share.png"); plt.show()

In [ ]:
# F4 WHO PQ finished products prequalified per year
p = pd.read_csv(T / "t6_pq_fpp_annual_by_area.csv")
p["grp"] = p["Therapeutic Area"].map({"HIV/AIDS": "HIV/ARV", "Tuberculosis": "Tuberculosis", "Malaria": "Malaria"}).fillna("Other")
w = p.pivot_table(index="year", columns="grp", values="n", aggfunc="sum").fillna(0).reindex(range(2002, 2027), fill_value=0)
order = ["HIV/ARV", "Tuberculosis", "Malaria", "Other"]
fig, ax = plt.subplots(figsize=(6.5, 4))
bottom = 0
for g in order:
    ax.bar(w.index, w[g], bottom=bottom, color=S[g], width=0.8, edgecolor="white", linewidth=1, label=g)
    bottom = bottom + w[g]
ax.set_xticks(range(2002, 2027, 2)); ax.grid(axis="x", visible=False)
ax.set_ylabel("Products prequalified"); ax.legend(frameon=False, ncol=4, loc="upper left", fontsize=11)
ax.set_ylim(0, bottom.max() * 1.18)
title(ax, "WHO prequalifications of currently listed finished products",
      "By year of prequalification and therapeutic area (n = 564 dated products)")
source(fig, "Source: WHO PQ medicines list, downloaded 2026-10-10. Only products still listed today; earlier years undercount (withdrawn products drop out).")
fig.savefig(F / "f4_pq_fpp_by_year.png"); plt.show()
print("ok")

Prequalifications of currently listed products rose from mostly under 15 per year before 2012 to about 40-50 per year in 2018-2024. Newer listings also include areas outside the three diseases, such as reproductive health, oncology and hepatitis. WHO only lists products that are still prequalified, so the early years undercount.

Chinese firms were more visible upstream than downstream. Shanghai Desano had 20 prequalified active ingredients, while Guilin was the only Chinese firm among the top finished-product suppliers in PQR.